In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [2]:
from pathlib import Path
import yaml

DATASET_ROOT = Path(
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final"
)

DATA_YAML = Path("/kaggle/working/visdrone_final.yaml")

names = [
    "pedestrian",
    "people",
    "bicycle",
    "car",
    "van",
    "truck",
    "tricycle",
    "awning-tricycle",
    "bus",
    "motor",
]

cfg = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 10,
    "names": names,
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Created:", DATA_YAML)
print("\nYAML contents:")
print(DATA_YAML.read_text())

# Verify required directories
print("\nDataset directory checks:")
for split in ["train", "val", "test"]:
    img_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    print(
        f"{split.upper():5s} | "
        f"images={img_dir.exists()} | "
        f"labels={label_dir.exists()}"
    )

Created: /kaggle/working/visdrone_final.yaml

YAML contents:
path: /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final
train: train/images
val: val/images
test: test/images
nc: 10
names:
- pedestrian
- people
- bicycle
- car
- van
- truck
- tricycle
- awning-tricycle
- bus
- motor


Dataset directory checks:
TRAIN | images=True | labels=True
VAL   | images=True | labels=True
TEST  | images=True | labels=True


In [3]:
from pathlib import Path
import yaml
import torch
from ultralytics import YOLO

SRC = Path(
    "/usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models/11/yolo11.yaml"
)

DST = Path(
    "/kaggle/working/yolo11n_visdrone_10class.yaml"
)

# Load official YOLO11 base architecture
with open(SRC, "r") as f:
    cfg = yaml.safe_load(f)

# Select Nano scale and VisDrone's 10 classes
cfg["scale"] = "n"
cfg["nc"] = 10

# Save controlled 10-class architecture
with open(DST, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("=" * 60)
print("YOLO11n VisDrone Architecture")
print("=" * 60)
print("Source:", SRC)
print("Created:", DST)
print("Scale:", cfg["scale"])
print("Classes:", cfg["nc"])

# Build model from YAML — no pretrained weights
model = YOLO(str(DST))

total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(
    p.numel() for p in model.model.parameters()
    if p.requires_grad
)

print("\nModel built successfully.")
print("Parameters:", f"{total_params:,}")
print("Trainable :", f"{trainable_params:,}")

# Verify detection head
for i, m in enumerate(model.model.model):
    if "Detect" in m.__class__.__name__:
        print("\nDetect layer:")
        print("Index:", i)
        print("Type :", m.__class__.__name__)
        print("Classes:", getattr(m, "nc", "N/A"))

# Dummy forward
x = torch.randn(1, 3, 640, 640)

with torch.no_grad():
    _ = model.model(x)

print("\nDummy forward: PASS")
print("=" * 60)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
YOLO11n VisDrone Architecture
Source: /usr/local/lib/python3.13/dist-packages/ultralytics/cfg/models/11/yolo11.yaml
Created: /kaggle/working/yolo11n_visdrone_10class.yaml
Scale: n
Classes: 10

Model built successfully.
Parameters: 2,591,790
Trainable : 2,591,774

Detect layer:
Index: 23
Type : Detect
Classes: 10

Dummy forward: PASS


In [4]:
from ultralytics import YOLO
from pathlib import Path
import torch
import ultralytics

MODEL_YAML = "/kaggle/working/yolo11n_visdrone_10class.yaml"
DATA_YAML = "/kaggle/working/visdrone_final.yaml"

PROJECT = "/kaggle/working/YOLO_VisDrone_Benchmark"
RUN_NAME = "YOLO11n_AdamW_seed42"

print("=" * 70)
print("YOLO11n | VisDrone | Controlled Baseline")
print("=" * 70)

print("Ultralytics :", ultralytics.__version__)
print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

print("\nProtocol")
print("  Model       : YOLO11n")
print("  Classes     : 10")
print("  Epochs      : 100")
print("  Image size  : 640")
print("  Batch       : 8")
print("  Optimizer   : AdamW")
print("  lr0         : 0.002")
print("  lrf         : 0.01")
print("  Weight decay: 0.0005")
print("  Cosine LR   : True")
print("  Warmup      : 3 epochs")
print("  AMP         : True")
print("  Seed        : 42")
print("  Determinism : True")
print("  Patience    : 20")
print("  Max det     : 1000")
print("  Workers     : 2")
print("  Copy-paste  : 0")
print("  Pretrained  : False")

model = YOLO(MODEL_YAML)

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,

    optimizer="AdamW",
    lr0=0.002,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,

    warmup_epochs=3,
    amp=True,

    seed=42,
    deterministic=True,

    patience=20,
    max_det=1000,

    workers=2,

    copy_paste=0,

    pretrained=False,

    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True
)

print("\n" + "=" * 70)
print("YOLO11n training completed")
print("=" * 70)

print(
    "Run directory:",
    f"{PROJECT}/{RUN_NAME}"
)

print(
    "Best checkpoint:",
    f"{PROJECT}/{RUN_NAME}/weights/best.pt"
)

YOLO11n | VisDrone | Controlled Baseline
Ultralytics : 8.4.211
PyTorch      : 2.11.0+cu130
CUDA         : True
GPU          : Tesla T4

Protocol
  Model       : YOLO11n
  Classes     : 10
  Epochs      : 100
  Image size  : 640
  Batch       : 8
  Optimizer   : AdamW
  lr0         : 0.002
  lrf         : 0.01
  Weight decay: 0.0005
  Cosine LR   : True
  Warmup      : 3 epochs
  AMP         : True
  Seed        : 42
  Determinism : True
  Patience    : 20
  Max det     : 1000
  Workers     : 2
  Copy-paste  : 0
  Pretrained  : False
Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/visdrone_final.yaml, degrees=0.0,

KeyboardInterrupt: 

In [3]:
from ultralytics import YOLO
runs = {  # name: path to best.pt
    "YOLOv10n": "/kaggle/input/notebooks/narendramodigari/yolov11n-drone-yolo-revision/YOLO_VisDrone_Benchmark/YOLO11n_AdamW_seed42/weights/best.pt",
    # add the other models here
}
for name, w in runs.items():
    YOLO(w).val(data="/kaggle/input/notebooks/narendramodigari/yolov11n-drone-yolo-revision/visdrone_final.yaml", split="test", imgsz=640,
                conf=0.001, iou=0.7, max_det=500, save_json=True, name=name)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n_visdrone_10class summary (fused): 100 layers, 2,584,102 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.8±0.6 ms, read: 2.3±1.4 MB/s, size: 189.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 18.2it/s 1:280.1sss
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/narendramodigari/visdrone-label-correc

In [4]:
# ============================================================
# SIZE-STRATIFIED AP / RECALL EVALUATION
# ============================================================

!pip install -q pycocotools pillow

import argparse
import csv
import json
from pathlib import Path

import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


BINS = {
    "coco": [
        ("small", 0, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],

    "fine": [
        ("tiny", 0, 16**2),
        ("small", 16**2, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],
}


def build_gt(img_dir, lbl_dir, n_classes, cat_offset):

    images = []
    anns = []
    aid = 1

    for p in sorted(Path(img_dir).iterdir()):

        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        w, h = Image.open(p).size

        iid = int(p.stem) if p.stem.isnumeric() else p.stem

        images.append(
            dict(
                id=iid,
                width=w,
                height=h,
                file_name=p.name
            )
        )

        lf = Path(lbl_dir) / (p.stem + ".txt")

        if not lf.exists():
            continue

        for line in lf.read_text().strip().splitlines():

            parts = line.split()

            if len(parts) < 5:
                continue

            c = int(float(parts[0]))

            xc, yc, bw, bh = (
                float(v) for v in parts[1:5]
            )

            # Convert normalized YOLO coordinates
            # to ORIGINAL image pixel coordinates
            bw = bw * w
            bh = bh * h

            x = xc * w - bw / 2
            y = yc * h - bh / 2

            anns.append(
                dict(
                    id=aid,
                    image_id=iid,
                    category_id=c + cat_offset,
                    bbox=[x, y, bw, bh],
                    area=bw * bh,
                    iscrowd=0
                )
            )

            aid += 1

    cats = [
        dict(
            id=i + cat_offset,
            name=str(i)
        )
        for i in range(n_classes)
    ]

    gt = COCO()

    gt.dataset = dict(
        images=images,
        annotations=anns,
        categories=cats
    )

    gt.createIndex()

    return gt


def mean_valid(a):

    a = a[a > -1]

    return float(a.mean()) if a.size else float("nan")


def evaluate(gt, pred_file, bins, maxdet):

    with open(pred_file, "r") as f:
        preds = json.load(f)

    dt = gt.loadRes(preds)

    E = COCOeval(
        gt,
        dt,
        "bbox"
    )

    E.params.maxDets = [10, 100, maxdet]

    E.params.areaRng = (
        [[0, 1e10]]
        + [[lo, hi] for _, lo, hi in bins]
    )

    E.params.areaRngLbl = (
        ["all"]
        + [b[0] for b in bins]
    )

    E.evaluate()
    E.accumulate()

    prec = E.eval["precision"]
    rec = E.eval["recall"]

    # maxDet index
    m = 2

    rows = []

    areas = np.array([
        a["area"]
        for a in gt.dataset["annotations"]
    ])

    for a, lbl in enumerate(E.params.areaRngLbl):

        lo, hi = E.params.areaRng[a]

        n = int(
            ((areas >= lo) & (areas <= hi)).sum()
        )

        rows.append(
            dict(
                bin=lbl,
                n_gt=n,

                AP50_95=mean_valid(
                    prec[:, :, :, a, m]
                ),

                AP50=mean_valid(
                    prec[0, :, :, a, m]
                ),

                AR50_95=mean_valid(
                    rec[:, :, a, m]
                ),

                AR50=mean_valid(
                    rec[0, :, a, m]
                )
            )
        )

    return rows


def main(argv=None):

    ap = argparse.ArgumentParser()

    ap.add_argument(
        "--images",
        required=True
    )

    ap.add_argument(
        "--labels",
        required=True
    )

    ap.add_argument(
        "--runs",
        nargs="+",
        required=True,
        help="NAME=path/to/predictions.json"
    )

    ap.add_argument(
        "--nc",
        type=int,
        default=10
    )

    ap.add_argument(
        "--bins",
        choices=BINS,
        default="fine"
    )

    ap.add_argument(
        "--maxdet",
        type=int,
        default=500
    )

    ap.add_argument(
        "--cat_offset",
        type=int,
        default=-1,
        help="0 or 1; -1 = infer from predictions"
    )

    ap.add_argument(
        "--out",
        default="size_results.csv"
    )

    a = ap.parse_args(argv)

    runs = dict(
        r.split("=", 1)
        for r in a.runs
    )

    # --------------------------------------------------------
    # Infer category offset
    # --------------------------------------------------------

    off = a.cat_offset

    if off < 0:

        first_file = next(
            iter(runs.values())
        )

        with open(first_file, "r") as f:
            first = json.load(f)

        off = min(
            p["category_id"]
            for p in first
        )

        print(
            "Inferred category offset:",
            off
        )

    # --------------------------------------------------------
    # Build ground truth
    # --------------------------------------------------------

    gt = build_gt(
        a.images,
        a.labels,
        a.nc,
        off
    )

    print(
        f"GT: {len(gt.dataset['images'])} images, "
        f"{len(gt.dataset['annotations'])} boxes"
    )

    # --------------------------------------------------------
    # Evaluate models
    # --------------------------------------------------------

    out = []

    for name, f in runs.items():

        print()
        print("=" * 70)
        print(f"Evaluating: {name}")
        print(f"Prediction file: {f}")
        print("=" * 70)

        for r in evaluate(
            gt,
            f,
            BINS[a.bins],
            a.maxdet
        ):

            r["model"] = name

            out.append(r)

            print(
                f"{name:12s} "
                f"{r['bin']:7s} "
                f"n={r['n_gt']:6d} "
                f"AP50={r['AP50']:.3f} "
                f"AP50-95={r['AP50_95']:.3f} "
                f"AR50={r['AR50']:.3f} "
                f"AR50-95={r['AR50_95']:.3f}"
            )

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    with open(
        a.out,
        "w",
        newline=""
    ) as fh:

        w = csv.DictWriter(
            fh,
            fieldnames=[
                "model",
                "bin",
                "n_gt",
                "AP50",
                "AP50_95",
                "AR50",
                "AR50_95"
            ]
        )

        w.writeheader()
        w.writerows(out)

    print()
    print("=" * 70)
    print("Saved:")
    print(a.out)
    print("=" * 70)


# ============================================================
# KAGGLE CONFIGURATION
# ============================================================

main([
    "--images",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/images",

    "--labels",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels",

    "--runs",

    "YOLOv11n=/kaggle/working/runs/detect/YOLOv10n/predictions.json",

    # "YOLOv9t=/kaggle/working/runs/detect/v9t/predictions.json",

    "--nc",
    "10",

    "--bins",
    "fine",

    "--maxdet",
    "500",

    "--cat_offset",
    "-1",

    "--out",
    "/kaggle/working/size_results.csv"
])

Inferred category offset: 1
creating index...
index created!
GT: 1610 images, 75102 boxes

Evaluating: YOLOv11n
Prediction file: /kaggle/working/runs/detect/YOLOv10n/predictions.json
Loading and preparing results...
DONE (t=3.67s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=210.31s).
Accumulating evaluation results...
DONE (t=13.59s).
YOLOv11n     all     n= 75102 AP50=0.230 AP50-95=0.125 AR50=0.542 AR50-95=0.290
YOLOv11n     tiny    n= 27346 AP50=0.071 AP50-95=0.025 AR50=0.236 AR50-95=0.088
YOLOv11n     small   n= 23491 AP50=0.180 AP50-95=0.074 AR50=0.548 AR50-95=0.249
YOLOv11n     medium  n= 21856 AP50=0.345 AP50-95=0.191 AR50=0.746 AR50-95=0.424
YOLOv11n     large   n=  2409 AP50=0.476 AP50-95=0.312 AR50=0.870 AR50-95=0.588

Saved:
/kaggle/working/size_results.csv
